# \[CDISC\] ADaM Analysis Data Model

SEOYEON CHOI  
2026-03-03

# ADaM Analysis Data Model

> The **ADaM (Analysis Data Model)** dataset is a CDISC standard for
> organizing clinical trial data to support statistical analysis,
> regulatory reporting, and traceability. Derived from **SDTM (Study
> Data Tabulation Model)** data, ADaM datasets are designed to be
> **“analysis-ready”** for generating tables, listings, and figures.
> They are required by **regulatory agencies like the FDA and PMDA**.

# Reference

-   [Data](https://github.com/cdisc-org/sdtm-adam-pilot-project/tree/master)
    -   [adamdata
        guide](https://github.com/cdisc-org/sdtm-adam-pilot-project/blob/master/updated-pilot-submission-package/900172/m5/datasets/cdiscpilot01/analysis/adam/datasets/dataguide.pdf)
-   [Explanation](https://www.lexjansen.com/pharmasug/2012/DS/PharmaSUG-2012-DS18.pdf)

# Import

In [1]:
import pandas as pd
import pyreadstat
import os
from pathlib import Path

# Data

In [2]:
data_folder = Path("../../../../delete/adam/")

In [3]:
files = [
    "adae.xpt",
    "adlbc.xpt",
    "adlbh.xpt",
    "adlbhy.xpt",
    "adqsadas.xpt",
    "adqscibc.xpt",
    "adqsnpix.xpt",
    "adsl.xpt",
    "adtte.xpt",
    "advs.xpt"
]

In [4]:
data = {}
metadata = {}

In [5]:
for f in files:
    full_path = data_folder / f
    df, meta = pyreadstat.read_xport(full_path)
    
    key = f.replace(".xpt","")
    
    data[key] = df
    metadata[key] = meta
    
    print(f"{f} loaded:", df.shape)

adae.xpt loaded: (1191, 55)
adlbc.xpt loaded: (74264, 46)
adlbh.xpt loaded: (49932, 46)
adlbhy.xpt loaded: (9954, 43)
adqsadas.xpt loaded: (12463, 40)
adqscibc.xpt loaded: (730, 36)
adqsnpix.xpt loaded: (31140, 41)
adsl.xpt loaded: (254, 48)
adtte.xpt loaded: (254, 26)
advs.xpt loaded: (32139, 34)

## ADSL

> The first is ADSL (Subject Level Analysis Dataset), a
> one-recordper-subject structure that contains subject-level
> attributes. Because of its structure, it can be merged onto any other
> clinical dataset, including other ADaM datasets and SDTM datasets.

In [6]:
data['adsl'].head()

In [7]:
metadata['adsl'].column_names_to_labels

# Table

## Table 11-1. Demographic Characteristics

In [289]:
data2 = data['adsl'].copy()

data2['RACE2'] = data2['RACE'].replace({
    'WHITE': 'White/Caucasian',
    'BLACK OR AFRICAN AMERICAN': 'Other',
    'AMERICAN INDIAN OR ALASKA NATIVE': 'Other'
})

In [290]:
n_counts = data['adsl'].groupby('ARM')['USUBJID'].nunique()
total_n = data['adsl']['USUBJID'].nunique()
n_counts['Total'] = total_n

In [291]:
def mean_min_max_table(df, var, group_col='ARM', total_label='Total',
                       mean_digits=1, minmax_digits=0):
    """
    Create Mean (Min–Max) summary table by group + Total
    
    Parameters
    ----------
    df : DataFrame
    var : str (continuous variable name)
    group_col : str (grouping column, default='ARM')
    total_label : str
    mean_digits : int (decimal places for mean)
    minmax_digits : int (decimal places for min/max)
    """

    # ARM별 요약
    arm_summary = (
        df.groupby(group_col)[var]
          .agg(['mean', 'min', 'max'])
    )

    # Total 요약
    total_summary = (
        df[var]
          .agg(['mean', 'min', 'max'])
          .to_frame().T
    )
    total_summary.index = [total_label]

    # 합치기
    final = pd.concat([arm_summary, total_summary])

    # 포맷팅
    final[f'{var} Mean (Min–Max)'] = (
        final['mean'].round(mean_digits).astype(str)
        + " ("
        + final['min'].round(minmax_digits).astype(int).astype(str)
        + "–"
        + final['max'].round(minmax_digits).astype(int).astype(str)
        + ")"
    )

    return final[[f'{var} Mean (Min–Max)']]

In [292]:
def categorical_table(df, var, group_col='ARM', total_label='Total', pct_digits=0):
    """
    Create n (%) table for categorical variable by group + Total
    """

    # 교차표
    ct = pd.crosstab(df[var], df[group_col])

    # ARM 기준 %
    pct = ct.div(ct.sum(axis=0), axis=1) * 100

    formatted = (
        ct.astype(str) + " (" +
        pct.round(pct_digits).astype(int).astype(str) + "%)"
    )

    # Total 계산
    total_n = df.shape[0]
    total_ct = df[var].value_counts()
    total_pct = total_ct / total_n * 100

    formatted[total_label] = (
        total_ct.astype(str) + " (" +
        total_pct.round(pct_digits).astype(int).astype(str) + "%)"
    )

    formatted.columns.name = None

    return formatted

In [313]:
Table11_1 = pd.concat([mean_min_max_table(data['adsl'],'AGE').T,
                    categorical_table(data['adsl'],'SEX').reset_index().set_index('SEX').reindex(['M', 'F']),
                    categorical_table(data2,'RACE2').reset_index().set_index('RACE2').reindex(['White/Caucasian', 'Other']),
                    mean_min_max_table(data['adsl'],'EDUCLVL').T,
                   ])

In [314]:
Table11_1.columns = [
    f"{col} (n={n_counts[col]})" if col in n_counts.index
    else col
    for col in final2.columns
]

In [315]:
Table11_1

In [9]:
data['adqscibc']

In [10]:
metadata['adqscibc'].column_names_to_labels

9.7.1.1. Endpoints - The primary efficacy endpoints were: - Alzheimer’s
Disease Assessment Scale - Cognitive Subscale, total of 11 items
\[ADAS-Cog (11)\] at Week 24 - Video-referenced Clinician’s
Interview-based Impression of Change (CIBIC+) at Week 24 - The secondary
efficacy endpoints were: - Alzheimer’s Disease Assessment Scale -
Cognitive Subscale, total of 11 items \[ADAS-Cog (11)\] at Weeks 8 and
16 - Video-referenced Clinician’s Interview-based Impression of Change
(CIBIC+) at Weeks 8 and 16 - Mean Revised Neuropsychiatric Inventory
(NPI-X) from Week 4 to Week 24 - The safety endpoints were: - Adverse
events - Vital signs (weight, standing and supine blood pressure, heart
rate) - Laboratory evaluations